# 一些API的使用

### 从etherscan查询手上的3000个和propose相关的contracts是否开源

In [4]:
import requests
import time
import os
import json
import ast

ETHERSCAN_API_KEY = "XT28VFFFF8CFYISGIZ57V6Y1IR85UW8VUX"
BASE_URL = "https://api.etherscan.io/v2/api"

def is_verified_contract(address, max_retries=3):
    params = {
        "chainid": "1",
        "module": "contract",
        "action": "getabi",
        "address": address,
        "apikey": ETHERSCAN_API_KEY
    }

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=10)
            data = resp.json()
            if data.get("status") == "1":
                return True
            return False
        except Exception as e:
            print(f"⚠️ 第 {attempt} 次尝试失败: {address}, 错误: {e}")
            time.sleep(2)

    print(f"❌ 最终失败: {address}")
    return False

def classify_contracts(addresses, output_dir="../../proxy_data/ES_proposal_data/contracts_classified", delay=0.2):
    os.makedirs(output_dir, exist_ok=True)
    verified_path = os.path.join(output_dir, "verified.txt")
    non_verified_path = os.path.join(output_dir, "non_verified.txt")

    # 加载已保存的地址，避免重复检查
    verified_done = set()
    non_verified_done = set()
    if os.path.exists(verified_path):
        with open(verified_path, "r") as f:
            verified_done = set(line.strip() for line in f)
    if os.path.exists(non_verified_path):
        with open(non_verified_path, "r") as f:
            non_verified_done = set(line.strip() for line in f)

    total = len(addresses)
    for idx, addr in enumerate(addresses):
        addr = addr.lower()
        if addr in verified_done or addr in non_verified_done:
            continue

        print(f"🔢 正在处理第 {idx + 1} / {total} 个：{addr}")
        verified = is_verified_contract(addr)
        if verified:
            print(f"✅ VERIFIED: {addr}")
            with open(verified_path, "a") as vf:
                vf.write(addr + "\n")
        else:
            print(f"❌ NOT VERIFIED: {addr}")
            with open(non_verified_path, "a") as nvf:
                nvf.write(addr + "\n")

        time.sleep(delay)

    print(f"\n✅ 所有合约已分类写入 {output_dir}/")

# === 加载地址列表 ===
contracts = []
with open('../../proxy_data/ES_proposal_data/propose_profiles/gov_contract_to_hashes.json', 'r', encoding='utf-8') as f:
    contract_data = json.load(f)
    for key in contract_data.keys():
        if key.startswith("[") and key.endswith("]"):
            addresses = ast.literal_eval(key)
            for address in addresses:
                contracts.append(address.lower())
        else:
            contracts.append(key.lower())

classify_contracts(contracts)

🔢 正在处理第 1 / 3804 个：0xe02640be68df835aa3327ea6473c02c8f6c3815a
❌ NOT VERIFIED: 0xe02640be68df835aa3327ea6473c02c8f6c3815a
🔢 正在处理第 2 / 3804 个：0x776a192f558f1b36329db47d4381778b825998e3
❌ NOT VERIFIED: 0x776a192f558f1b36329db47d4381778b825998e3
🔢 正在处理第 3 / 3804 个：0x8ed4b8b8859e247f3e10eae876485cf2bc8abcf7
❌ NOT VERIFIED: 0x8ed4b8b8859e247f3e10eae876485cf2bc8abcf7
🔢 正在处理第 4 / 3804 个：0x7bb0b08587b8a6b8945e09f1baca426558b0f06a
✅ VERIFIED: 0x7bb0b08587b8a6b8945e09f1baca426558b0f06a
🔢 正在处理第 5 / 3804 个：0x8a9a37391fabc28faa169a0e7cb20d883794eb52
❌ NOT VERIFIED: 0x8a9a37391fabc28faa169a0e7cb20d883794eb52
🔢 正在处理第 6 / 3804 个：0x5be9887d5b4336cb762a9d73335991a6ff8ce981
✅ VERIFIED: 0x5be9887d5b4336cb762a9d73335991a6ff8ce981
🔢 正在处理第 7 / 3804 个：0x7d37011b1a5c3f7c276c9957558a8347676b4877
❌ NOT VERIFIED: 0x7d37011b1a5c3f7c276c9957558a8347676b4877
🔢 正在处理第 8 / 3804 个：0x2305576962e33102b01d8f3dfda2fa640137b15e
❌ NOT VERIFIED: 0x2305576962e33102b01d8f3dfda2fa640137b15e
🔢 正在处理第 9 / 3804 个：0xee62d2a0420a1a5322f

### 从4byte拿所有 propose() 的 hex

In [ ]:
import requests
import time
import json

def fetch_all_signatures(text_signature, save_path="propose_signatures.json", delay=0.5):
    base_url = "https://www.4byte.directory/api/v1/signatures/"
    params = {"text_signature": text_signature}
    all_results = []

    print(f"🔍 开始查询: {text_signature}")
    while True:
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"❌ 请求失败: {response.status_code}")
            break

        data = response.json()
        results = data.get("results", [])
        all_results.extend(results)

        print(f"📄 获取 {len(results)} 条记录，累计 {len(all_results)} 条")

        if not data.get("next"):
            break
        else:
            # 下一页的 URL 已经包含所有参数了，所以直接覆盖
            base_url = data["next"]
            params = {}
            time.sleep(delay)

    # 保存到本地
    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"✅ 所有结果已保存至: {save_path}")

# 使用示例
fetch_all_signatures("propose(")

### 从4byte拿所有 execute() 的 hex

In [ ]:
import json

def filter_execute_like_signatures(input_file, output_file):
    with open(input_file, "r") as f:
        data = json.load(f)

    filtered = [
        entry for entry in data
        if isinstance(entry.get("text_signature"), str) and (
            entry["text_signature"].startswith("execute(") or
            entry["text_signature"].startswith("batchExecute(")
        )
    ]

    print(f"✅ 共找到 {len(filtered)} 条以 execute( 或 batchExecute( 开头的 text_signature")

    with open(output_file, "w") as f:
        json.dump(filtered, f, indent=2)

    print(f"✅ 已保存筛选结果至: {output_file}")

# 使用示例
filter_execute_like_signatures("execute_signatures.json", "execute_sig.json")

In [17]:
import os
import shutil

def load_verified_addresses(verified_path):
    with open(verified_path, "r") as f:
        return set(line.strip().lower() for line in f)

def organize_contract_files(base_dir="../../proxy_data/ES_proposal_data/contract_propose_execute/non_verified/execute/", verified_path="../../proxy_data/ES_proposal_data/contracts_classified/verified.txt"):
    verified_addrs = load_verified_addresses(verified_path)

    # 输出目录结构
    output_dirs = {
        "verified_propose": os.path.join("verified", "propose"),
        "verified_execute": os.path.join("verified", "execute"),
        "non_verified_propose": os.path.join("non_verified", "propose"),
        "non_verified_execute": os.path.join("non_verified", "execute"),
    }

    # 创建目录
    for path in output_dirs.values():
        os.makedirs(path, exist_ok=True)

    # 遍历 base_dir 下所有文件
    for filename in os.listdir(base_dir):
        full_path = os.path.join(base_dir, filename)

        if not os.path.isfile(full_path):
            continue

        if filename.startswith("propose_"):
            addr = filename[len("propose_"):-5].lower()
            if addr in verified_addrs:
                dst = os.path.join(output_dirs["verified_propose"], filename)
                dst = os.path.join("../../proxy_data/ES_proposal_data/contract_propose_execute/", dst)
                shutil.move(full_path, dst)
                print(f"📂 移动 {filename} 到 {dst}")
        elif filename.startswith("execute_"):
            addr = filename[len("execute_"):-5].lower()
            if addr in verified_addrs:
                dst = os.path.join(output_dirs["verified_execute"], filename)
                dst = os.path.join("../../proxy_data/ES_proposal_data/contract_propose_execute/", dst)
                shutil.move(full_path, dst)
                print(f"📂 移动 {filename} 到 {dst}")
        else:
            continue  # 跳过不是 propose_ 或 execute_ 开头的文件


    print("✅ 文件分类完成")

# 执行脚本
organize_contract_files()

📂 移动 execute_0x3af8009b82b7fd54e8e22c19a1170e4d28869e86.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0x3af8009b82b7fd54e8e22c19a1170e4d28869e86.json
📂 移动 execute_0xe7b6c199bbb1b4c08efdadd4bbe99d246ac9a7dd.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0xe7b6c199bbb1b4c08efdadd4bbe99d246ac9a7dd.json
📂 移动 execute_0x5d2c31ce16924c2a71d317e5bbfd5ce387854039.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0x5d2c31ce16924c2a71d317e5bbfd5ce387854039.json
📂 移动 execute_0xf1750b770485a5d0589a6ba1270d9fc354884d45.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0xf1750b770485a5d0589a6ba1270d9fc354884d45.json
📂 移动 execute_0x099b06f471dfcbb025b5212d6507895709e8134a.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0x099b06f471dfcbb025b5212d6507895709e8134a.json
📂 移动 execute_0x8582d1e7d3

In [19]:
verified_path="../../proxy_data/ES_proposal_data/contracts_classified/verified.txt"
non_verified_path="../../proxy_data/ES_proposal_data/contracts_classified/non_verified.txt"
verified_addrs = load_verified_addresses(verified_path)
print(len(verified_addrs))

non_verified_addrs = load_verified_addresses(non_verified_path)
print(len(non_verified_addrs))

print(len(verified_addrs) + len(non_verified_addrs))

1103
2578
3681


In [3]:
import json
def load_data(json_path):
    with open(json_path, "r") as f:
        return json.load(f)

if __name__ == "__main__":
    json_path = "../../proxy_data/ES_proposal_data/contract_propose_execute/proposal_vs_execute_summary.json"
    data = load_data(json_path)
    filtered_data = [entry for entry in data if entry["execute_count"] > 0]
    print(f"✅ 筛选后共有 {len(filtered_data)} 条记录，包含 execute_count > 0 的合约")
    
    # 复制所有 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_<filtered_data[i]>.json 到文件夹 ../../proxy_data/ES_proposal_data/contract_propose_execute/gov_exe_prop/
    import os
    import shutil
    output_dir = "../../proxy_data/ES_proposal_data/contract_propose_execute/gov_exe_prop/"
    os.makedirs(output_dir, exist_ok=True)
    for entry in filtered_data:
        address = entry["address"]
        filename = f"execute_{address}.json"
        src_path = os.path.join("../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/", filename)
        dst_path = os.path.join(output_dir, filename)
        
        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
            print(f"✅ 复制 {src_path} 到 {dst_path}")
        else:
            print(f"❌ 文件不存在: {src_path}")
            
        filename = f"propose_{address}.json"
        src_path = os.path.join("../../proxy_data/ES_proposal_data/contract_propose_execute/verified/propose/", filename)
        dst_path = os.path.join(output_dir, filename)
        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
            print(f"✅ 复制 {src_path} 到 {dst_path}")
        else:
            print(f"❌ 文件不存在: {src_path}")

✅ 筛选后共有 854 条记录，包含 execute_count > 0 的合约
✅ 复制 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0x001e6bf774191e71f617d90adc6af7a792a8bf77.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/gov_exe_prop/execute_0x001e6bf774191e71f617d90adc6af7a792a8bf77.json
✅ 复制 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/propose/propose_0x001e6bf774191e71f617d90adc6af7a792a8bf77.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/gov_exe_prop/propose_0x001e6bf774191e71f617d90adc6af7a792a8bf77.json
✅ 复制 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/execute/execute_0x00cf5d86a3755b82d78c516da8ff2a0883cb3b75.json 到 ../../proxy_data/ES_proposal_data/contract_propose_execute/gov_exe_prop/execute_0x00cf5d86a3755b82d78c516da8ff2a0883cb3b75.json
✅ 复制 ../../proxy_data/ES_proposal_data/contract_propose_execute/verified/propose/propose_0x00cf5d86a3755b82d78c516da8ff2a0883cb3b75.json 到 ../../proxy_data/ES

In [2]:
# 重新导入依赖并重新执行之前的代码块
import json
import ast
import os
from opensearchpy import OpenSearch


# 修改后的 ContractESQuery 支持按合约地址和函数类型筛选，并保存结果为 JSON
class ContractESQuery:
    def __init__(self, block_time, call_functions, contract_address, host='192.168.3.146', port=9200):

        auth = (os.getenv('OPENSEARCH_USER'), os.getenv('OPENSEARCH_PASSWORD'))
        self.index_name = 'eth_block'
        self.block_time = block_time
        self.call_functions = call_functions
        self.contract_address = contract_address.lower()
        self.client = OpenSearch(
            hosts=[{'host': host, 'port': port}],
            http_compress=True,
            http_auth=auth,
            use_ssl=False,
            verify_certs=False,
            ssl_assert_hostname=False,
            ssl_show_warn=False
        )
        self.dsl_query = self.build_query()

    def build_query(self):
        return {
            "_source": False,
            "size": 20000,
            "query": {
                "bool": {
                    "filter": [
                        {
                            "range": {
                                "Timestamp": {
                                    "gte": f"{self.block_time-10}-01-01",
                                    "lte": f"{self.block_time}-12-31"
                                }
                            }
                        },
                        {
                            "nested": {
                                "path": "Transactions",
                                "inner_hits": {
                                    "_source": [
                                        "Transactions.Hash",
                                        "Transactions.CallFunction",
                                        "Transactions.CallParameter",
                                        "Transactions.FromAddress",
                                        "Transactions.ToAddress",
                                        "Transactions.Logs",
                                        "Transactions.InternalTxns.CallFunction",
                                        "Transactions.InternalTxns.CallParameter",
                                        "Transactions.InternalTxns.FromAddress",
                                        "Transactions.InternalTxns.ToAddress",
                                        "Transactions.InternalTxns.Type"
                                    ],
                                    "size": 100
                                },
                                "query": {
                                    "bool": {
                                        "should": [
                                            {
                                                "bool": {
                                                    "must": [
                                                        {"terms": {"Transactions.CallFunction": self.call_functions}},
                                                        {"term": {"Transactions.ToAddress": self.contract_address}}
                                                    ]
                                                }
                                            },
                                            {
                                                "nested": {
                                                    "path": "Transactions.InternalTxns",
                                                    "query": {
                                                        "bool": {
                                                            "must": [
                                                                {"terms": {
                                                                    "Transactions.InternalTxns.CallFunction": self.call_functions
                                                                }},
                                                                {"term": {
                                                                    "Transactions.InternalTxns.ToAddress": self.contract_address
                                                                }}
                                                            ]
                                                        }
                                                    }
                                                }
                                            }
                                        ],
                                        "minimum_should_match": 1
                                    }
                                }
                            }
                        }
                    ]
                }
            }
        }

    def get_results(self):
        response = self.client.search(index=self.index_name, body=self.dsl_query, timeout=300)
        results = []
        for hit in response['hits']['hits']:
            inner_hits = hit.get('inner_hits', {}).get('Transactions', {}).get('hits', {}).get('hits', [])
            for tx in inner_hits:
                tx_src = tx.get('_source', {})
                tx_hash = tx_src.get('Hash')
                tx_block_number = tx.get('_id')

                if tx_src.get('CallFunction') in self.call_functions and tx_src.get('ToAddress', '').lower() == self.contract_address:
                    results.append({
                        'BlockNumber': tx_block_number,
                        'CallFunction': tx_src.get('CallFunction'),
                        'CallParameter': tx_src.get('CallParameter'),
                        'FromAddress': tx_src.get('FromAddress'),
                        'ToAddress': tx_src.get('ToAddress'),
                        'Hash': tx_hash,
                        'Type': 'External',
                        'Logs': tx_src.get('Logs', []),
                    })

                for internal in tx_src.get('InternalTxns', []):
                    if internal.get('CallFunction') in self.call_functions and internal.get('ToAddress', '').lower() == self.contract_address:
                        results.append({
                            "BlockNumber": tx_block_number,
                            'CallFunction': internal.get('CallFunction'),
                            'CallParameter': internal.get('CallParameter'),
                            'FromAddress': internal.get('FromAddress'),
                            'ToAddress': internal.get('ToAddress'),
                            'Hash': tx_hash,
                            'Type': 'Internal',
                            'InternalType': internal.get('Type', 'Unknown'),
                            'Logs': tx_src.get('Logs', []),
                            'InternalTxns': internal
                        })
        return results

# 保存函数：按合约地址写入 JSON 文件
def save_txns_by_contract(results, contract_address):
    filename = f"{contract_address}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ 已保存 {len(results)} 条交易到 {filename}")

if __name__ == "__main__":
    # 示例调用
    year = 2025
    
    hex_function_propose_group1 = ["0xa25632fd"]
    hex_function_execute_group1 = ["0xf0689b47"]
    hex_function_propose_group2 = ["0x7d5e81e2", "0xda95691a", "0x490145c8"]
    hex_function_execute_group2 = ["0xfe0d94c1", "0x2656227d", "0x60e69a7b"]
    propose_signatures = hex_function_propose_group1 + hex_function_propose_group2
    execute_signatures = hex_function_execute_group1 + hex_function_execute_group2
    
    # contract = "0xe02640be68df835aa3327ea6473c02c8f6c3815a"
    with open('../../proxy_data/ES_proposal_data/contract_propose_execute/all_intersected_contracts.json', 'r', encoding='utf-8') as f:
        contract_data = json.load(f)
    
    for contract in contract_data:
        print(f"Processing contract: {contract}")
        
        query = ContractESQuery(block_time=year, call_functions=propose_signatures, contract_address=contract)
        results = query.get_results()
        save_txns_by_contract(results, f"../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/propose_{contract}")
        
        query = ContractESQuery(block_time=year, call_functions=execute_signatures, contract_address=contract)
        results = query.get_results()
        save_txns_by_contract(results, f"../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/execute_{contract}")

Processing contract: 0xfc5bbcb00a58ea37ffaf5ad29609afcea359b6e0
✅ 已保存 1 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/propose_0xfc5bbcb00a58ea37ffaf5ad29609afcea359b6e0.json
✅ 已保存 1 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/execute_0xfc5bbcb00a58ea37ffaf5ad29609afcea359b6e0.json
Processing contract: 0x52b4489b71caa0ddf14db121b25d0a4d360af575
✅ 已保存 30 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/propose_0x52b4489b71caa0ddf14db121b25d0a4d360af575.json
✅ 已保存 26 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/execute_0x52b4489b71caa0ddf14db121b25d0a4d360af575.json
Processing contract: 0xf0adebf82d6fe3c91b2bd5efca87ab17d3792ad9
✅ 已保存 1 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop/propose_0xf0adebf82d6fe3c91b2bd5efca87ab17d3792ad9.json
✅ 已保存 1 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4

In [5]:
import json

with open('../../proxy_data/ES_proposal_data/contract_propose_execute/all_intersected_contracts.json', 'r', encoding='utf-8') as f:
    contract_data = json.load(f)
print(len(contract_data))
print(contract_data[:10])  # 打印前10个合约地址

808
['0xfc5bbcb00a58ea37ffaf5ad29609afcea359b6e0', '0x52b4489b71caa0ddf14db121b25d0a4d360af575', '0xf0adebf82d6fe3c91b2bd5efca87ab17d3792ad9', '0x4a3e36a9f70bfe565550a17e45bb1fe6fb712f56', '0x7ae22bebf28366c328d5558e6fad935487299dfe', '0x94aeb84caa2ba26a198dcc7bdc43cb92cfa88bb4', '0x3ab7bd4480ce1f0d6c6d2f985a64fbffe623874e', '0x78669f5366bcd0b0c10926e56a2434510674313e', '0x111484bd069d513a4d0c4019f02ae9b9c396d0fc', '0x640418ad05a8251cf01e9eba182abe4cf883819c']


In [1]:
import json
from pathlib import Path
import sys

# === 项目根目录路径 ===
PROJECT_ROOT = "../"
sys.path.append(PROJECT_ROOT)

# === 工具函数导入 ===
from utils.abi_decoder import decode_transaction_input
from utils.event_decoder import decode_event_log

# === 输入输出路径 ===
input_dir = Path("../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop")
output_dir = Path("../../proxy_data/ES_proposal_data/contract_propose_execute/top4_sig_exe_prop_decoded")
output_dir.mkdir(parents=True, exist_ok=True)

# === 遍历解码 ===
for file in input_dir.glob("propose_*.json"):
    try:
        with open(file, "r") as f:
            data = json.load(f)

        for entry in data:
            if "CallFunction" in entry and "CallParameter" in entry:
                entry["DecodedCall"] = decode_transaction_input(entry["CallFunction"], entry["CallParameter"])
            if "Logs" in entry:
                for log in entry["Logs"]:
                    if log.get("Topics"):
                        log["DecodedLog"] = decode_event_log(log["Topics"][0], log["Topics"], log["Data"])

        output_path = output_dir / file.name
        with open(output_path, "w") as f:
            json.dump(data, f, indent=2)

        print(f"[✓] {file.name} decoded.")

    except Exception as e:
        print(f"[✗] {file.name} failed: {e}")

[✓] propose_0xa09cd9d9287580b7c417b07494347b9f7d596cc7.json decoded.
[✓] propose_0x8505fb8e97ae7434f610c8851df7cf7831353bd1.json decoded.
[✓] propose_0xeca77dbfc28b63d1c72d096e69ed9cd3da1cc486.json decoded.
[✓] propose_0x7a6bbe7fdd793cc9ab7e0fc33605fcd2d19371e8.json decoded.
[✓] propose_0xe410defa07cb8b7dce0b3064d88c144b959d6215.json decoded.
[✓] propose_0xc0da01a04c3f3e0be433606045bb7017a7323e38.json decoded.
[✓] propose_0x8869a94df9200c75116a285e12e85c24179129e1.json decoded.
[✓] propose_0x177702231005a46ce1e921900702cde96c9c9319.json decoded.
[✓] propose_0x5e5031627408fc2a75c8560f9c84548c1de6fe37.json decoded.
[✓] propose_0x7499a7dd98989dd3ee34dcd4961f642c114963ae.json decoded.
[✓] propose_0x30bf81c54488096ac75e3adc83ee9d6cb18a266e.json decoded.
[✓] propose_0x6853f8865ba8e9fbd9c8cce3155ce5023fb7eeb0.json decoded.
[✓] propose_0x3f7274d3f58ccc6d00fc2ad0935cce9ad4e6e24b.json decoded.
[✓] propose_0x0add6d42bbfe6c40e15b02a2c8a1b81b36a2b326.json decoded.
[✓] propose_0x441808e20e625e0094b0

* contract: 0xCDb9F8f9bE143B7c72480185459AB9720462a786 LootDAO 用的 castVote 接口和别人不一样, 一会儿记得重新跑一下

signature: castVote (0xb5079e5d)

* 0x74d5b005ca64a5C9EE3611Bdc6F6C02D93C84b2f 这个的接口很奇怪, 一会儿要另行设置一下

* 0x7725c6045092DEB243ae23aFB4355f43aAC1FC22, 这个都适用voteWithReason投的, 可以最后可以手动加载一下voter

* 0xc35df7360d5783988481af9365405694ed3fc408 同上

* 0x748bA9Cd5a5DDba5ABA70a4aC861b2413dCa4436 同上

* 0xC101D4F9C78B089Ea6EE6F31Dc5a3D4cf4C19270 同上

*其余的基本都可以说是private DAO 总共71 - 6 = 65个不存在投票的contract*


In [3]:
# 重新导入依赖并重新执行之前的代码块
import json
import ast
import os
from opensearchpy import OpenSearch


# 修改后的 ContractESQuery 支持按合约地址和函数类型筛选，并保存结果为 JSON
class ContractESQuery:
    def __init__(self, block_time, call_functions, contract_address, host='192.168.3.146', port=9200):

        auth = (os.getenv('OPENSEARCH_USER'), os.getenv('OPENSEARCH_PASSWORD'))
        self.index_name = 'eth_block'
        self.block_time = block_time
        self.call_functions = call_functions
        self.contract_address = contract_address.lower()
        self.client = OpenSearch(
            hosts=[{'host': host, 'port': port}],
            http_compress=True,
            http_auth=auth,
            use_ssl=False,
            verify_certs=False,
            ssl_assert_hostname=False,
            ssl_show_warn=False
        )
        self.dsl_query = self.build_query()

    def build_query(self):
        return {
            "_source": False,
            "size": 20000,
            "query": {
                "bool": {
                    "filter": [
                        {
                            "range": {
                                "Timestamp": {
                                    "gte": f"{self.block_time-10}-01-01",
                                    "lte": f"{self.block_time}-12-31"
                                }
                            }
                        },
                        {
                            "nested": {
                                "path": "Transactions",
                                "inner_hits": {
                                    "_source": [
                                        "Transactions.Hash",
                                        "Transactions.CallFunction",
                                        "Transactions.CallParameter",
                                        "Transactions.FromAddress",
                                        "Transactions.ToAddress",
                                        "Transactions.Logs",
                                        "Transactions.InternalTxns.CallFunction",
                                        "Transactions.InternalTxns.CallParameter",
                                        "Transactions.InternalTxns.FromAddress",
                                        "Transactions.InternalTxns.ToAddress",
                                        "Transactions.InternalTxns.Type"
                                    ],
                                    "size": 100
                                },
                                "query": {
                                    "bool": {
                                        "should": [
                                            {
                                                "bool": {
                                                    "must": [
                                                        {"terms": {"Transactions.CallFunction": self.call_functions}},
                                                        {"term": {"Transactions.ToAddress": self.contract_address}}
                                                    ]
                                                }
                                            },
                                            {
                                                "nested": {
                                                    "path": "Transactions.InternalTxns",
                                                    "query": {
                                                        "bool": {
                                                            "must": [
                                                                {"terms": {
                                                                    "Transactions.InternalTxns.CallFunction": self.call_functions
                                                                }},
                                                                {"term": {
                                                                    "Transactions.InternalTxns.ToAddress": self.contract_address
                                                                }}
                                                            ]
                                                        }
                                                    }
                                                }
                                            }
                                        ],
                                        "minimum_should_match": 1
                                    }
                                }
                            }
                        }
                    ]
                }
            }
        }

    def get_results(self):
        response = self.client.search(index=self.index_name, body=self.dsl_query, timeout=300)
        results = []
        for hit in response['hits']['hits']:
            inner_hits = hit.get('inner_hits', {}).get('Transactions', {}).get('hits', {}).get('hits', [])
            for tx in inner_hits:
                tx_src = tx.get('_source', {})
                tx_hash = tx_src.get('Hash')
                tx_block_number = tx.get('_id')

                if tx_src.get('CallFunction') in self.call_functions and tx_src.get('ToAddress', '').lower() == self.contract_address:
                    results.append({
                        'BlockNumber': tx_block_number,
                        'CallFunction': tx_src.get('CallFunction'),
                        'CallParameter': tx_src.get('CallParameter'),
                        'FromAddress': tx_src.get('FromAddress'),
                        'ToAddress': tx_src.get('ToAddress'),
                        'Hash': tx_hash,
                        'Type': 'External',
                        'Logs': tx_src.get('Logs', []),
                    })

                for internal in tx_src.get('InternalTxns', []):
                    if internal.get('CallFunction') in self.call_functions and internal.get('ToAddress', '').lower() == self.contract_address:
                        results.append({
                            "BlockNumber": tx_block_number,
                            'CallFunction': internal.get('CallFunction'),
                            'CallParameter': internal.get('CallParameter'),
                            'FromAddress': internal.get('FromAddress'),
                            'ToAddress': internal.get('ToAddress'),
                            'Hash': tx_hash,
                            'Type': 'Internal',
                            'InternalType': internal.get('Type', 'Unknown'),
                            'Logs': tx_src.get('Logs', []),
                            'InternalTxns': internal
                        })
        return results

# 保存函数：按合约地址写入 JSON 文件
def save_txns_by_contract(results, contract_address):
    filename = f"{contract_address}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ 已保存 {len(results)} 条交易到 {filename}")

if __name__ == "__main__":
    # 示例调用
    year = 2025
    
    # hex_function_propose_group1 = ["0xa25632fd"]
    # hex_function_execute_group1 = ["0xf0689b47"]
    # hex_function_propose_group2 = ["0x7d5e81e2", "0xda95691a", "0x490145c8"]
    # hex_function_execute_group2 = ["0xfe0d94c1", "0x2656227d", "0x60e69a7b"]
    # propose_signatures = hex_function_propose_group1 + hex_function_propose_group2
    # execute_signatures = hex_function_execute_group1 + hex_function_execute_group2
    vote_signatures = ["0x09358479", "0x75a12d72", "0x56781388", "0xdf18e809", "0x15373e3d", "0xb5079e5d", "0x41f9b62c", "0xc36560b8", "0x7b3c71d3"]
    
    # with open('../../proxy_data/ES_proposal_data/contract_propose_execute/all_intersected_contracts.json', 'r', encoding='utf-8') as f:
    #     contract_data = json.load(f)
    
    contract_data = ["0xCDb9F8f9bE143B7c72480185459AB9720462a786", "0x74d5b005ca64a5C9EE3611Bdc6F6C02D93C84b2f", "0x7725c6045092DEB243ae23aFB4355f43aAC1FC22", "0xc35df7360d5783988481af9365405694ed3fc408", "0x748bA9Cd5a5DDba5ABA70a4aC861b2413dCa4436", "0xC101D4F9C78B089Ea6EE6F31Dc5a3D4cf4C19270"]
    
    for contract in contract_data:
        contract = contract.lower()
        print(f"Processing contract: {contract}")
        query = ContractESQuery(block_time=year, call_functions=vote_signatures, contract_address=contract)
        results = query.get_results()
        save_txns_by_contract(results, f"../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_{contract}")

Processing contract: 0xcdb9f8f9be143b7c72480185459ab9720462a786
✅ 已保存 245 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_0xcdb9f8f9be143b7c72480185459ab9720462a786.json
Processing contract: 0x74d5b005ca64a5c9ee3611bdc6f6c02d93c84b2f
✅ 已保存 75 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_0x74d5b005ca64a5c9ee3611bdc6f6c02d93c84b2f.json
Processing contract: 0x7725c6045092deb243ae23afb4355f43aac1fc22
✅ 已保存 37 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_0x7725c6045092deb243ae23afb4355f43aac1fc22.json
Processing contract: 0xc35df7360d5783988481af9365405694ed3fc408
✅ 已保存 4 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_0xc35df7360d5783988481af9365405694ed3fc408.json
Processing contract: 0x748ba9cd5a5ddba5aba70a4ac861b2413dca4436
✅ 已保存 1 条交易到 ../../proxy_data/ES_proposal_data/contract_propose_execute/top4_vote/vote_0x748ba9cd5a5ddba5aba70a4ac861b2413dca4436.json
P